# 04 Leakage-safe feature engineering
Rule: a feature for a payment at time *t* may use only events with time < *t*. We implement this with one primitive (`EventIndex`) and prove it with a **prefix-invariance test**: features computed on the first half of the data must equal those computed with all data.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
from presage.features.windows import EventIndex
import numpy as np
# toy: three events of one account at t=10,20,20 (values 1,2,4). A query at t=20 may only see the t=10 event.
ev = EventIndex(np.array([1,1,1]), np.array([10.,20.,20.]), dict(v=np.array([1.,2.,4.])))
print('sum before t=20:', ev.sum('v', np.array([1]), np.array([20.]), np.inf), '(expected 1.0)')

sum before t=20: [1.] (expected 1.0)


In [3]:
from presage.data.loaders import load_sim
from presage.features.temporal import build_features
df, acc = load_sim(); X, groups = build_features(df, acc)
print({k: len(v) for k, v in groups.items()}, 'total', X.shape[1]-1)
print(groups['beneficiary'][:12])

18:45:18 INFO features: built 90 temporal features for 390498 rows


{'txn': 9, 'behaviour': 46, 'beneficiary': 30, 'device': 5} total 90
['new_beneficiary', 'prior_pays_to_dst', 'dst_age_days', 'dst_age_lt30d', 'cross_bank', 'dst_in_cnt_1h', 'dst_in_sum_1h', 'dst_fanin_new_1h', 'dst_in_cnt_24h', 'dst_in_sum_24h', 'dst_fanin_new_24h', 'dst_in_cnt_7d']


### The leakage test (the same logic is in `tests/test_leakage.py`)

In [4]:
cut = df.ts.quantile(.5); pre = df[df.ts < cut].copy(); Xp, _ = build_features(pre, acc)
a = X.iloc[:len(pre)].drop(columns='txn_id').values; b = Xp.drop(columns='txn_id').values
bad = [c for j, c in enumerate(X.columns[1:]) if not np.allclose(a[:, j], b[:, j], equal_nan=True, atol=1e-3)]
print('rows compared:', len(pre), '| features that change when the future is removed:', bad)

18:45:20 INFO features: built 90 temporal features for 195249 rows


rows compared: 195249 | features that change when the future is removed: []


Empty list = no feature uses the future. (An earlier version of the rolling std failed this test only through floating-point cancellation; it was fixed with group-centred prefix sums and a variance floor.)

Windows: 10 min, 1 h, 6 h, 24 h, 7 d, 30 d with count, sum, mean, std, max and new-beneficiary count; plus recipient novelty, time-of-day deviation, balance share, device/IP novelty, and beneficiary-side fan-in, fan-out, pass-through and holding time.

In [5]:
x = X.assign(y=df.is_app_fraud.values)[df.src_is_customer.values==1]
x.groupby('y')[['pct_balance','amt_to_hist_mean','new_beneficiary','dst_age_days','dst_fanin_new_24h','dst_pass_through_24h','session_z','new_device']].mean().T.round(3)

y,0,1
pct_balance,0.037000,0.164000
amt_to_hist_mean,1.294000,20.836000
new_beneficiary,0.224000,0.595000
dst_age_days,580.224976,654.497009
dst_fanin_new_24h,4.144000,0.246000
dst_pass_through_24h,0.037000,0.298000
session_z,0.051000,2.114000
new_device,0.023000,0.058000
